In [6]:
# Imports and Configuration
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [7]:
# ---- EDIT THIS PATH ----
INPUT_PATH = "/home/jovyan/work/Cardiology_Data/miriad_cardiology.json"
# ------------------------

In [8]:
OUTPUT_TRAIN = "hyde_train_split.jsonl"
OUTPUT_VAL = "hyde_val_split.jsonl"

In [9]:
# Load, Filter, and Balance Data
print("Loading raw dataset...")
df = pd.read_json(INPUT_PATH)
print(f"Total rows loaded: {len(df)}")

Loading raw dataset...
Total rows loaded: 276139


In [10]:
# Filter for Cardiology and Cardiothoracic Surgery, Vascular Surgery
df_filtered = df[df["specialty"].isin(["Cardiology", "Cardiothoracic Surgery", "Vascular Surgery"])]
print(f"Filtered (Cardiology + Cardiothoracic Surgery, Vascular Surgery): {len(df_filtered)}")

# Use entire filtered dataset (no sampling)
balanced_df = df_filtered.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Final dataset size: {len(balanced_df)}")

Filtered (Cardiology + Cardiothoracic Surgery, Vascular Surgery): 276139
Final dataset size: 276139


In [11]:
# Format for T5 HyDE Training
# We map Question -> Answer (The "Answer" acts as our Hypothetical Document)

print("Formatting data for model...")

# T5 expects specific column names for some scripts, but we'll use 'input_text' and 'target_text'
# Prefix: "hypothetical answer: " tells the model what task we are doing
df_formatted = pd.DataFrame({
    "input_text": "hypothetical answer: " + balanced_df["question"].astype(str),
    "target_text": balanced_df["answer"].astype(str)
})

# Preview
print("\nSample Training Pair:")
print("INPUT :", df_formatted.iloc[0]["input_text"])
print("TARGET:", df_formatted.iloc[0]["target_text"][:200] + "...") # Truncate for display

Formatting data for model...

Sample Training Pair:
INPUT : hypothetical answer: What is the role of TNFα in cardiac tissue of patients with chronic heart failure?

TARGET: TNFα is found to be increased in cardiac tissue of patients with chronic heart failure. It is suggested that the failing heart is the cause of immune activation, leading to increased expression of TNF...


In [12]:
# Split and Save
# 90% Training, 10% Validation
train_df, val_df = train_test_split(df_formatted, test_size=0.10, random_state=42, shuffle=True)

print(f"\nTraining set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")

# Save to JSONL
train_df.to_json(OUTPUT_TRAIN, orient="records", lines=True)
val_df.to_json(OUTPUT_VAL, orient="records", lines=True)

print(f"\nSaved files:\n1. {OUTPUT_TRAIN}\n2. {OUTPUT_VAL}")
print("Ready for fine-tuning!")


Training set size: 248525
Validation set size: 27614

Saved files:
1. hyde_train_split.jsonl
2. hyde_val_split.jsonl
Ready for fine-tuning!
